# Case Study 2 — FINAL run, fully local (generator + judge on the uni GPU)

One local open generator, one local open judge, no API keys, no cost.

- **Generator:** `microsoft/phi-4` (14B, MIT licence, not gated, Microsoft family, not a reasoning model). Served with vLLM.
- **Judge:** `Qwen/Qwen2.5-72B-Instruct-AWQ` (the locked judge config). Served with vLLM.
- Both pinned to the exact Hugging Face commit, greedy decoding. Revisions are written to `results/run_manifest.json`.
- Same 215 rows, same retrieval, same prompts. All three configs by default (generation is cheap on a local model).

**Order:** generator up → generate → generator down → checkpoint zip → judge up → score → judge down → report → final zip.

**The sandbox wipes on close.** Download `results/runs_checkpoint.zip` as soon as section 7 finishes, and `results/FINAL_RESULTS.zip` at the end.
If the session dies mid-generation, rerun from the top: generation uses `--resume` and only runs the missing rows (as long as `results/runs/` still exists).

Run from the repo root.

In [ ]:
!pkill -f vllm ; sleep 5 ; nvidia-smi   # kill any orphan vLLM server first

## 0. Environment probe  (must show an A6000 / 48 GB card)

In [ ]:
import sys, os, subprocess, platform, urllib.request, json, time, glob, datetime
print("python:", sys.version.split()[0], "|", platform.platform())
print("cwd:", os.getcwd())
assert os.path.exists("src/run_generation.py"), "Run this notebook from the repo ROOT."
try:
    urllib.request.urlopen("https://pypi.org", timeout=5); HAS_INTERNET = True
except Exception as e:
    HAS_INTERNET = False; print("internet check failed:", e)
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} | {p.total_memory/1e9:.0f} GB")
print(f"\nSUMMARY  internet={HAS_INTERNET}  gpu={HAS_GPU}")
assert HAS_GPU, "CPU-only node. Restart the server on the GPU profile (2x A6000)."

## 1. Install dependencies (once)

In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], check=False)
subprocess.run([sys.executable,"-m","pip","install","-q","openai","vllm","huggingface_hub"], check=False)
print("deps installed")

## 2. Config — the only knobs

In [ ]:
# ---------- GENERATOR (local) ----------
GEN_MODEL      = "microsoft/phi-4"
GEN_PORT       = 8001
GEN_MAX_TOKENS = 2048     # generous so truncation is never OUR cap's fault
GEN_MAX_LEN    = 8192     # context window given to vLLM (prompt ~2.5k tokens + output)

# ---------- JUDGE (local, locked) ----------
JUDGE_MODEL      = "Qwen/Qwen2.5-72B-Instruct-AWQ"
JUDGE_PORT       = 8000
JUDGE_MAX_TOKENS = 2048   # room for the per-statement verdict JSON
JUDGE_MAX_LEN    = 6144   # locked value (fits one A6000)

# ---------- RUN ----------
CONFIGS   = ["baseline1_plain_llm", "baseline2_standard_rag", "agent_structured"]
# short on GPU time? use ["agent_structured"] only
WORKERS   = 16            # concurrent requests; vLLM batches them
THRESHOLD = 0.5           # faithful cut for the bucket matrix (a sweep is reported too)
LIMIT     = None          # e.g. 8 for a quick smoke test, None for all 215

# ---------- pin exact revisions ----------
from huggingface_hub import model_info
GEN_REV   = model_info(GEN_MODEL).sha
JUDGE_REV = model_info(JUDGE_MODEL).sha
os.makedirs("results", exist_ok=True)
manifest = {
    "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
    "generator": {"model": GEN_MODEL, "revision": GEN_REV, "temperature": 0.0, "top_p": 1.0,
                  "max_tokens": GEN_MAX_TOKENS, "server": "vLLM", "max_model_len": GEN_MAX_LEN},
    "judge": {"model": JUDGE_MODEL, "revision": JUDGE_REV, "temperature": 0.0,
              "max_tokens": JUDGE_MAX_TOKENS, "server": "vLLM", "max_model_len": JUDGE_MAX_LEN},
    "configs": CONFIGS, "faithful_threshold": THRESHOLD, "limit": LIMIT,
}
json.dump(manifest, open("results/run_manifest.json","w"), indent=2)
print(json.dumps(manifest, indent=2))

def sh(cmd):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout: print(r.stdout[-6000:])
    if r.returncode != 0:
        print("STDERR:\n", r.stderr[-4000:])
        raise RuntimeError(f"step failed (exit {r.returncode}): {' '.join(cmd[:3])} ... scroll up for STDERR")
    return r.returncode

VLLM_ENV = {**os.environ, "VLLM_USE_FLASHINFER_SAMPLER": "0"}   # container can't JIT FlashInfer

def start_vllm(model, rev, port, max_len, util=0.90, log="vllm.log"):
    proc = subprocess.Popen([sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model, "--revision", rev, "--port", str(port), "--dtype", "auto",
        "--gpu-memory-utilization", str(util), "--max-model-len", str(max_len),
        "--served-model-name", model],
        env=VLLM_ENV, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    print(f"starting vLLM {model}@{rev[:10]} on :{port}  (log: {log})")
    for i in range(360):                      # up to 60 min for a first download
        if proc.poll() is not None:
            print("vLLM DIED. Last log lines:"); print(open(log).read()[-3000:]); return None
        try:
            urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=3)
            print(f"vLLM up after ~{i*10}s"); return proc
        except Exception:
            time.sleep(10)
    print("vLLM did not come up in time. Check", log); return proc

def stop_vllm(proc):
    if proc is None: return
    proc.terminate()
    try: proc.wait(timeout=60)
    except Exception: proc.kill()
    time.sleep(10)
    subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv"])
    print("vLLM stopped, GPU memory freed")

## 2b. Preflight: are the patched scripts on this server?

In [ ]:
need = {"src/run_generation.py": "--resume", "src/run_scoring.py": "--workers",
        "src/faithfulness.py": "_safe_score", "src/retrieval_eval.py": "retrieval_eval_out"}
missing = [f for f, marker in need.items() if marker not in open(f).read()]
for f in ["config/pipeline_redacted.yaml", "test_set_redacted.jsonl"]:
    if not os.path.exists(f): missing.append(f)
assert not missing, f"OLD or MISSING files: {missing} -> unzip case_study2_all_changes.zip over the repo root"
print("preflight OK: patched scripts and redacted data present")

## 3. Corpus (400 committed chunks)

In [ ]:
CHUNKS = "results/corpus_chunks.jsonl"
print("corpus chunks:", sum(1 for _ in open(CHUNKS)))

## 4. Embed + index (bge, Chroma)

In [ ]:
sh([sys.executable, "src/embed_and_index.py", "config/pipeline.yaml"])

## 5. Retrieval check (expect Hit@5 ≈ 0.833)

In [ ]:
sh([sys.executable, "src/retrieval_eval.py", "config/pipeline.yaml"])

## 6. Generate with the local generator

In [ ]:
gen_proc = start_vllm(GEN_MODEL, GEN_REV, GEN_PORT, GEN_MAX_LEN, util=0.90, log="vllm_generator.log")
assert gen_proc is not None
t0 = time.time()
gen_args = [sys.executable, "src/run_generation.py",
            "--provider", "openai_compatible", "--model", GEN_MODEL,
            "--base-url", f"http://localhost:{GEN_PORT}/v1", "--api-key-env", "NO_KEY_NEEDED",
            "--max-tokens", str(GEN_MAX_TOKENS), "--workers", str(WORKERS),
            "--resume", "--configs", *CONFIGS]
if LIMIT: gen_args += ["--limit", str(LIMIT)]
sh(gen_args)
print(f"generation took {(time.time()-t0)/60:.1f} min")
stop_vllm(gen_proc)

for f in sorted(glob.glob("results/runs/*.jsonl")):
    rows = [json.loads(l) for l in open(f)]
    bad  = sum(1 for r in rows if not r["parse_ok"])
    err  = sum(1 for r in rows if (r.get("usage") or {}).get("error"))
    empty= sum(1 for r in rows if not (r.get("raw") or "").strip())
    print(f"{os.path.basename(f):32s} rows={len(rows)}  parse_fail={bad}  empty_raw={empty}  call_errors={err}")

## 7. CHECKPOINT — download `results/runs_checkpoint.zip` now

In [ ]:
import shutil
shutil.make_archive("results/runs_checkpoint", "zip", "results", "runs")
print("wrote results/runs_checkpoint.zip  ->  right-click > Download in the file browser")

## 8. Judge + scoring (Qwen2.5-72B-AWQ, single card)

In [ ]:
judge_proc = start_vllm(JUDGE_MODEL, JUDGE_REV, JUDGE_PORT, JUDGE_MAX_LEN, util=0.95, log="vllm_judge.log")
assert judge_proc is not None
t0 = time.time()
sh([sys.executable, "src/run_scoring.py", "--judge", "vllm",
    "--judge-model", JUDGE_MODEL, "--judge-base-url", f"http://localhost:{JUDGE_PORT}/v1",
    "--judge-max-tokens", str(JUDGE_MAX_TOKENS), "--workers", str(WORKERS),
    "--faithful-threshold", str(THRESHOLD), "--configs", *CONFIGS])
print(f"scoring took {(time.time()-t0)/60:.1f} min")
stop_vllm(judge_proc)

## 9. Results

In [ ]:
for f in ["results/scoring/correctness_summary.md",
          "results/scoring/faithfulness_summary.md",
          "results/scoring/buckets_summary.md"]:
    print("="*72); print(f); print("="*72)
    print(open(f).read() if os.path.exists(f) else "(not produced)"); print()

## 10. Report + final zip — download `results/FINAL_RESULTS.zip`

In [ ]:
sh([sys.executable, "src/make_report.py", "--generator", f"{GEN_MODEL} (local, vLLM)", "--max-cases", "10"])
for log in ["vllm_generator.log", "vllm_judge.log"]:
    if os.path.exists(log): shutil.copy(log, "results/")
import zipfile
with zipfile.ZipFile("results/FINAL_RESULTS.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["results/runs/*.jsonl", "results/scoring/**/*", "results/report.html",
                "results/run_manifest.json", "results/retrieval_eval.*", "results/vllm_*.log"]:
        for p in glob.glob(pat, recursive=True):
            if os.path.isfile(p): z.write(p)
print("wrote results/FINAL_RESULTS.zip  ->  download it before closing the session")